# Speech translation with ESPnet-ST-v2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Courses/CMUSpeechTechnology26S/speech_translation.ipynb) [![speech_translation](https://github.com/espnet/notebook/actions/workflows/speech_translation.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/speech_translation.yml)

Translate speech into text offline, then do it simultaneously with a streaming
model, and score both with BLEU and the latency metrics SimulEval reports.

This is the demonstration part of an assignment from CMU 11492/11692/18495,
*Speech Technology for Conversational AI*, kept here without the graded
exercises so that it runs end to end.

Main references:
- [ESPnet repository](https://github.com/espnet/espnet)
- [ESPnet-ST-v2 paper](https://arxiv.org/abs/2304.04596)
- [ESPnet-ST recipe template](https://github.com/espnet/espnet/tree/master/egs2/TEMPLATE/st1)


## Install

We use the inference install of ESPnet rather than the full one: it carries no
training stack, so it is much smaller and much faster to install.


In [ ]:
%pip install -q "espnet==202610.post1"
!pip install -q espnet_model_zoo
!pip install transformers

We also have some other toolkits/packages needed for this assignment.


In [ ]:
!pip install --upgrade --no-cache-dir gdown
!git clone --depth 1 https://github.com/kan-bayashi/ParallelWaveGAN.git
!cd ParallelWaveGAN && pip install .
!pip install pysndfile
!pip install sacrebleu
!pip install mosestokenizer
!git clone https://github.com/facebookresearch/SimulEval.git
!cd SimulEval && pip install -e .

## Speech Translation

Speech translation is a typical task that translate speech in a language into text/speech in another language. In this tutorial, we will show you the some latest models (in ESPnet-ST-v2) in the field of speech translation and demonstrate using them in different scenarios, including

- offline speech-to-text translation
- simultaneous speech-to-text translation
- speech-to-speech translation


## Overview of the ESPnet-ST-v2

ESPnet-ST-v2 is a revamp of the open-source ESPnet-ST toolkit necessitated by the broadening interests of the spoken language translation community.
ESPnet-ST-v2 supports 1) offline speech-to-text translation (ST), 2) simultaneous speech-to-text translation (SST), and 3) offline speech-to-speech translation (S2ST) -- each task is supported with a wide variety of approaches, differentiating ESPnet-ST-v2 from other open source spoken language translation toolkits.
This toolkit offers state-of-the-art architectures such as transducers, hybrid CTC/attention, multi-decoders with searchable intermediates, time-synchronous blockwise CTC/attention, Translatotron models, and direct discrete unit models.

![picture](https://drive.google.com/uc?id=1taR-6Cq4akhhq3oQtgR7Y9mWD0vaEm4z)

In general, the toolkit is organizd in a pythonic way to support model training/inference, while we also provide recipes for data preparation, model training, and evaluation.

![pitcture](https://drive.google.com/uc?id=1I3w9BAYBhyaf440pBr6VX885f7oJN3mp)


## 1. Offline Speech-to-text Translation (ST)


### 1.1 Model download


In [ ]:
# The only copy of this en-es model is on Google Drive: the ESPnet
# organisation publishes en-de MuST-C models, not this one. If the
# link ever rots, the recipe that trained it is egs2/must_c_v2/st1.
!gdown 1Sn2rAZXVSm1hrCj5OIlq61EgbjKXNGdq
!unzip -o st_train_st_ctc_md_conformer_asrinit_v3_noamp_batch50m_ctcsamp0.1_lr1e-3_raw_en_es_bpe_tc4000_sp_valid.acc.ave.zip

### 1.2 Model Setup


In [ ]:
import time
import torch
import string
from espnet2.bin.st_inference import Speech2Text

lang="es"
fs = 16000

speech2text = Speech2Text(
    st_model_file="exp/st_train_st_ctc_md_conformer_asrinit_v3_noamp_batch50m_ctcsamp0.1_lr1e-3_raw_en_es_bpe_tc4000_sp/valid.acc.ave_10best.pth",
    st_train_config="exp/st_train_st_ctc_md_conformer_asrinit_v3_noamp_batch50m_ctcsamp0.1_lr1e-3_raw_en_es_bpe_tc4000_sp/config.yaml",
    beam_size=10,
    ctc_weight=0.3,
    asr_beam_size=10,
    asr_ctc_weight=0.3,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

### 1.3 Translate our example recordings


In [ ]:
!git clone https://github.com/ftshijt/ESPnet_st_egs.git

In [ ]:
import torch
import pandas as pd
import soundfile as sf
import librosa.display
from IPython.display import display, Audio
import matplotlib.pyplot as plt
from sacrebleu.metrics import BLEU

bleu = BLEU()

egs = pd.read_csv("ESPnet_st_egs/st/egs.csv")
for index, row in egs.iterrows():
  if row["lang"] == lang or lang == "multilingual":
    speech, rate = sf.read("ESPnet_st_egs/" + row["path"])
    assert fs == int(row["sr"])
    text, _, _, _ = speech2text(speech)[0][0]
    display(Audio(speech, rate=fs))
    librosa.display.waveshow(speech, sr=fs, color="blue")
    plt.show()
    print(f"Reference source text: {row['src_text']}")
    print(f"Translation results: {text}")
    print(f"Reference target text: {row['tgt_text']}")
    print(f"Sentence BLEU Score: {bleu.sentence_score(text, [row['tgt_text']])}")
    print("*" * 50)


### Score the translations with BLEU

The cell above printed a sentence BLEU per example. Corpus BLEU over the whole
set is the number a paper would report, and it is not the average of those.


In [ ]:
hyps, refs = [], [[]]
for _, row in egs.iterrows():
    if row["lang"] == lang or lang == "multilingual":
        speech, _ = sf.read("ESPnet_st_egs/" + row["path"])
        text, _, _, _ = speech2text(speech)[0][0]
        hyps.append(text)
        refs[0].append(row["tgt_text"])

print(bleu.corpus_score(hyps, refs))


## 2. Simultaneous Speech-to-text Translation (SST)


In [ ]:
# Likewise for the streaming model: Drive is the only copy.
!gdown 1ekUeMvmaB3ZhAIY_KAb_we1zhIRFZhtu
!unzip -o st_train_st_ctc_conformer_asrinit_v2_streaming_40block_nohier_18lyr_raw_en_es_bpe_tc4000_sp_valid.acc.ave.zip

In [ ]:
import time
import torch
import string
from espnet2.bin.st_inference_streaming import Speech2TextStreaming

lang="es"
fs = 16000

speech2textstreaming = Speech2TextStreaming(
    st_model_file="exp/st_train_st_ctc_conformer_asrinit_v2_streaming_40block_nohier_18lyr_raw_en_es_bpe_tc4000_sp/valid.acc.ave_10best.pth",
    st_train_config="exp/st_train_st_ctc_conformer_asrinit_v2_streaming_40block_nohier_18lyr_raw_en_es_bpe_tc4000_sp/config.yaml",
    penalty=0.4,
    blank_penalty=0.5,
    beam_size=10,
    ctc_weight=0.5,
    incremental_decode=True,
    time_sync=True,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

In [ ]:
import torch
import pandas as pd
import soundfile as sf
import librosa.display
from IPython.display import display, Audio
import matplotlib.pyplot as plt
from sacrebleu.metrics import BLEU

bleu = BLEU()

egs = pd.read_csv("ESPnet_st_egs/st/egs.csv")
for index, row in egs.iterrows():
  if row["lang"] == lang or lang == "multilingual":
    speech, rate = sf.read("ESPnet_st_egs/" + row["path"])
    assert fs == int(row["sr"])
    text = speech2textstreaming(speech)[0][0]
    display(Audio(speech, rate=fs))
    librosa.display.waveshow(speech, sr=fs, color="blue")
    plt.show()
    print(f"Reference source text: {row['src_text']}")
    print(f"Translation results: {text}")
    print(f"Reference target text: {row['tgt_text']}")
    print(f"Sentence BLEU Score: {bleu.sentence_score(text, [row['tgt_text']])}")
    print("*" * 50)

### 2.1 Running SST with SimulEval


In [ ]:
!simuleval --source ESPnet_st_egs/st/wav.scp --target ESPnet_st_egs/st/ref.detok.trn --agent espnet/egs2/TEMPLATE/st1/pyscripts/utils/simuleval_agent.py --batch_size 1 --ngpu 0 --st_train_config exp/st_train_st_ctc_conformer_asrinit_v2_streaming_40block_nohier_18lyr_raw_en_es_bpe_tc4000_sp/config.yaml --st_model_file exp/st_train_st_ctc_conformer_asrinit_v2_streaming_40block_nohier_18lyr_raw_en_es_bpe_tc4000_sp/valid.acc.ave_10best.pth --disable_repetition_detection false --beam_size 10 --sim_chunk_length 2048 --backend streaming --ctc_weight 0.5 --incremental_decode true --penalty 0.4 --blank_penalty 0.7 --time_sync true --latency-metrics LAAL AL AP DAL